Structure to 3DFace - PV Riodades
=
This notebook reads the string blocks of the PV Riodades fixed-structure module drawing (`módulos.dwg`) and writes an AutoCAD script that re-inserts **one block per structure**: `EMF309_string_2` for a 2-high structure (two stacked strings) and `EMF309_string_1` for a single-string structure. The tag attributes of the top string are copied to the new block.

**Workflow overview:**
1. Convert the DWG to DXF and load its entities into the `cad_entities` table of `pv_projects.duckdb` as project `"PV Riodades"`.
2. Query all the entities of the project.
3. Build the target layer name by stripping the last 7 characters of each layer name.
4. Classify each string as top (`T`) or bottom (`B`) of a stack, or single (`C`).
5. Drop the bottom strings and choose the block size (2 for `T`, 1 for `C`).
6. Write `Reinserting Trackers - PV Riodades.scr`, grouped by target layer.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `módulos.dwg` sits in `dwg_folder` (`C:\Users\Usuario\Desktop\projetos\riodades\eletromec`). The DXF is written to the same folder and deleted at the end. The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`. The converter converts every DWG in `temp_folder`, so keep only this drawing in it.
- **Database:** `load_to_duckdb` deletes every existing `"PV Riodades"` row in `cad_entities` before loading.
- **Model space only, and only string blocks:**
  - The query takes every entity of the project, with no block name or entity type filter.
  - The drawing must contain nothing but the string block references (`INSERT`) in model space. Any other entity (line, text, another block) is treated as a string and gets an insert line (with empty coordinates if it isn't a block).
- **Stacking:**
  - The two strings of a 2-high structure have the same X (±0.5) and are less than 5 apart in Y.
  - Only single and 2-high structures are handled. In a 3-high stack, only the top string becomes a 2-string block and the other two are dropped.
- **Layers:**
  - Every layer name ends with a suffix of exactly 7 characters (e.g. `_F1_1.1`), and the rest is the target layer (`EMF_eletromec_estrutura_fixa_710Wp_F1_1.1` → `EMF_eletromec_estrutura_fixa_710Wp`).
  - A longer suffix (e.g. `_F1_1.10`) leaves a wrong layer name.
- **Attributes:** the string blocks carry `UFV`, `PST`, `INV`, `STR` and `GRP`. They are copied as text from the top string of each stack.
- **In the drawing where the script runs:**
  - The block definitions `EMF309_string_1` and `EMF309_string_2` must exist, with their base point matching the top string's insertion point. The block name comes from the stack size, not from the original block name.
  - The script fills 7 attributes positionally, in the order AutoCAD prompts for them: `UFV`, `PST`, `INV`, *(4th)*, `STR`, `GRP`, *(7th)*. The 4th and 7th get `""` (Enter), so they keep the block's default value.
  - The blocks have *Scale uniformly* on, so the insert asks for one scale factor (`1`). The rotation is always `0`, and the original rotation isn't read.
  - The target layers (e.g. `EMF_eletromec_estrutura_fixa_710Wp`) must already exist.
  - Set `ATTREQ = 1` and `ATTDIA = 0` so the attribute values are read from the script and not from a dialog.
  - The script only inserts. The original string blocks stay in the drawing unless you delete them.

Extracting the AutoCAD data and loading it into the DuckDB database
=
- Copies `módulos.dwg` into `temp_folder`. `convert_dwg_to_dxf` runs the ODA File Converter on that folder and writes the DXF (ACAD2018) into `output_folder` (= `dwg_folder`).
- `load_to_duckdb` reads every model space entity with `ezdxf` and loads it into `cad_entities` as `"PV Riodades"`. It deletes the project's old rows first. For blocks it stores the name, insertion point (`X`, `Y`), rotation, color and the attribute tags.
- The DXF and the temporary DWG are deleted. The connection `conn` stays open until the last cell.
- `numpy` is used for the block size step. `re` is imported but not used.

In [53]:
import numpy as np
import re

from DWG_to_DuckDB_ETL import *
import os
import shutil


dwg_folder = r"C:\Users\Usuario\Desktop\projetos\riodades\eletromec"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\riodades\eletromec\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "módulos.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "módulos.dxf")

temp_dwg_file = os.path.join(temp_folder, "módulos.dwg")

conn = load_to_duckdb(dxf_file, "PV Riodades")
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Riodades already exists
Removed existing data for project: PV Riodades
Successfully loaded project
Loaded 2858 entities for project: PV Riodades
Don't forget to close connection.
Data Loaded
DuckDB Loaded


Loading the data from PV Riodades
=
Selects `layer`, `name`, `X`, `Y`, `UFV`, `PST`, `INV`, `STR` and `GRP` for every entity of `PV Riodades`, with no name or type filter, so the drawing must only hold string blocks (2858 in the saved output). `rotation` isn't selected, because the script always inserts with rotation `0`.

In [54]:
structures = conn.sql("SELECT layer, name, X, Y, UFV, PST, INV, STR, GRP FROM cad_entities WHERE project_name = 'PV Riodades'")
structures.show()

┌───────────────────────────────────────────┬─────────────────┬───────────┬───────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│                   layer                   │      name       │     X     │     Y     │   UFV   │   PST   │   INV   │   STR   │   GRP   │
│                  varchar                  │     varchar     │   float   │   float   │ varchar │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────────────────┼─────────────────┼───────────┼───────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│ EMF_eletromec_estrutura_fixa_710Wp_F1_1.1 │ EMF309_string_2 │ 55192.176 │ 149897.77 │ 1       │ 1       │ 01      │ 01      │ 1.1.01  │
│ EMF_eletromec_estrutura_fixa_710Wp_F1_1.1 │ EMF309_string_2 │ 55192.176 │ 149895.44 │ 1       │ 1       │ 01      │ 02      │ 1.1.01  │
│ EMF_eletromec_estrutura_fixa_710Wp_F1_1.1 │ EMF309_string_2 │ 55231.867 │ 149897.77 │ 1       │ 1       │ 01      │ 03      │ 1.1.01  │
│ EMF_eletromec_estrutura_fixa_710

Converts the DuckDB relation into the pandas DataFrame `struct_df`.

In [55]:
struct_df = structures.to_df()
print(struct_df)

                                          layer             name  \
0     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
1     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
2     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
3     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
4     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
...                                         ...              ...   
2853  EMF_eletromec_estrutura_fixa_710Wp_F1_4.2  EMF309_string_1   
2854  EMF_eletromec_estrutura_fixa_710Wp_F1_4.2  EMF309_string_1   
2855  EMF_eletromec_estrutura_fixa_715Wp_F1_2.3  EMF309_string_1   
2856  EMF_eletromec_estrutura_fixa_710Wp_F2_3.1  EMF309_string_1   
2857  EMF_eletromec_estrutura_fixa_710Wp_F2_3.1  EMF309_string_1   

                 X              Y UFV PST INV STR     GRP  
0     55192.175781  149897.765625   1   1  01  01  1.1.01  
1     55192.175781  149895.437500   1   1  01  02  1.1.01  
2  

Target layer
=
`new_layer` = the layer name without its last 7 characters, which strips the `_Fn_U.P` suffix (`EMF_eletromec_estrutura_fixa_710Wp_F1_1.1` → `EMF_eletromec_estrutura_fixa_710Wp`). The new blocks are inserted on this layer. The suffix must be exactly 7 characters long.

In [56]:
struct_df['new_layer'] = struct_df['layer'].astype(str).str[:-7]

print(struct_df[['layer', 'new_layer']])

                                          layer  \
0     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1   
1     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1   
2     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1   
3     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1   
4     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1   
...                                         ...   
2853  EMF_eletromec_estrutura_fixa_710Wp_F1_4.2   
2854  EMF_eletromec_estrutura_fixa_710Wp_F1_4.2   
2855  EMF_eletromec_estrutura_fixa_715Wp_F1_2.3   
2856  EMF_eletromec_estrutura_fixa_710Wp_F2_3.1   
2857  EMF_eletromec_estrutura_fixa_710Wp_F2_3.1   

                               new_layer  
0     EMF_eletromec_estrutura_fixa_710Wp  
1     EMF_eletromec_estrutura_fixa_710Wp  
2     EMF_eletromec_estrutura_fixa_710Wp  
3     EMF_eletromec_estrutura_fixa_710Wp  
4     EMF_eletromec_estrutura_fixa_710Wp  
...                                  ...  
2853  EMF_eletromec_estrutura_fixa_710Wp  
2854  EMF_eletromec_estrutura_fixa_710Wp  


Stack classification
=
- For each string, it looks for other strings less than 0.5 away in X and 5 in Y, with a different Y (same structure column).
  - Above all of them → `T` (top of a 2-high structure). Otherwise → `B`.
  - None found → `C` (single-string structure).
- The loop runs row by row (`iterrows`) and compares against every string, so it gets slow on large drawings.
- In `value_counts()`, `T` and `B` must be equal (1342 each in the saved output), which means every stack pairs up.

In [57]:
for idx, row in struct_df.iterrows():
    x = row['X']
    y = row['Y']


    mask_TMB = (
        (struct_df['Y'] >= y - 5) &
        (struct_df['Y'] <= y + 5) &
        (struct_df['X'] >= x - 0.5) &
        (struct_df['X'] <= x + 0.5) &
        (struct_df['Y'] != y) &
        (struct_df.index != idx)
    )

    if mask_TMB.any():
        y_neighbors = struct_df.loc[mask_TMB, 'Y']

        if (y > y_neighbors).all():
            struct_df.loc[idx, 'Type'] = "T"
        else:
            struct_df.loc[idx, 'Type'] = "B"

    else:
        struct_df.loc[idx, 'Type'] = "C"

print(struct_df)
print(struct_df['Type'].value_counts())

                                          layer             name  \
0     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
1     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
2     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
3     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
4     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
...                                         ...              ...   
2853  EMF_eletromec_estrutura_fixa_710Wp_F1_4.2  EMF309_string_1   
2854  EMF_eletromec_estrutura_fixa_710Wp_F1_4.2  EMF309_string_1   
2855  EMF_eletromec_estrutura_fixa_715Wp_F1_2.3  EMF309_string_1   
2856  EMF_eletromec_estrutura_fixa_710Wp_F2_3.1  EMF309_string_1   
2857  EMF_eletromec_estrutura_fixa_710Wp_F2_3.1  EMF309_string_1   

                 X              Y UFV PST INV STR     GRP  \
0     55192.175781  149897.765625   1   1  01  01  1.1.01   
1     55192.175781  149895.437500   1   1  01  02  1.1.01   


Block size
=
- Drops the `B` strings, so each 2-high structure is represented only by its top string.
- `Group Size` = 2 for `T` and 1 for `C`. It picks the block `EMF309_string_2` or `EMF309_string_1`.
- The attributes of the bottom string (its `STR`, and its `INV` / `GRP` if they differ) are lost.
- `not_b_structs_df` is a filtered slice without `.copy()`, so pandas may show a `SettingWithCopyWarning` here and in the next step.

In [58]:
not_b_structs_df = struct_df[struct_df['Type'] != "B"]
not_b_structs_df['Group Size'] = np.where(not_b_structs_df['Type'] == "T", 2, 1)
print(not_b_structs_df)
print(not_b_structs_df['Type'].value_counts())

                                          layer             name  \
0     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
2     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
4     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
6     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
8     EMF_eletromec_estrutura_fixa_710Wp_F1_1.1  EMF309_string_2   
...                                         ...              ...   
2853  EMF_eletromec_estrutura_fixa_710Wp_F1_4.2  EMF309_string_1   
2854  EMF_eletromec_estrutura_fixa_710Wp_F1_4.2  EMF309_string_1   
2855  EMF_eletromec_estrutura_fixa_715Wp_F1_2.3  EMF309_string_1   
2856  EMF_eletromec_estrutura_fixa_710Wp_F2_3.1  EMF309_string_1   
2857  EMF_eletromec_estrutura_fixa_710Wp_F2_3.1  EMF309_string_1   

                 X              Y UFV PST INV STR     GRP  \
0     55192.175781  149897.765625   1   1  01  01  1.1.01   
2     55231.867188  149897.765625   1   1  01  03  1.1.01   


Writing the AutoCAD script
=
- One AutoLISP line per structure:
  `(command-s "._insert" "EMF309_string_<size>" "X,Y" 1 0 "UFV" "PST" "INV" "" "STR" "GRP" "")`
  - The coordinates are rounded to 4 decimals. `1` is the (uniform) scale and `0` is the rotation.
  - The 4th and 7th attributes get `""` (Enter), so they keep the block's default value.
- **Reinserting Trackers - PV Riodades.scr** (hardcoded path under `Desktop\Pessoal\Data Engineering\Python\Ecotechne\3dface`, overwritten each run): for each `new_layer`, it first sets that layer as current (`._layer set`) and then writes all the insert lines for it.
- Run it with `SCRIPT` in AutoCAD, with `ATTREQ = 1` and `ATTDIA = 0`.

In [63]:
not_b_structs_df['Script'] = not_b_structs_df.apply(lambda row: (
        f'(command-s "._insert" "EMF309_string_{row["Group Size"]}" '
        f'"{round(row["X"],4)},{round(row["Y"],4)}" 1 0 "{row["UFV"]}" "{row["PST"]}" "{row["INV"]}" "" "{row["STR"]}" "{row["GRP"]}" "")'
        ),
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\3dface\Reinserting Trackers - PV Riodades.scr", 'w') as f:
    for i in not_b_structs_df['new_layer'].unique():
        mask = not_b_structs_df['new_layer'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(not_b_structs_df[mask]['Script'].astype(str)) + '\n')

Closes the DuckDB connection.

In [ ]:
conn.close()